In [1]:
from pathlib import Path
import json

from tqdm.auto import tqdm

from pydantic import BaseModel, Field
from typing import Literal
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)


In [2]:
class PrinciplesMetadata(BaseModel):

    topic: list[
        Literal[
            "program_design",
            "periodization",
            "progressive_overload",
            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "recovery",
            "fatigue",
            "warmup",
            "energy_systems",
        ]
    ] = Field(min_length=1)

    planner_stage: list[
        Literal[
            "goal_selection",
            "program_design",
            "exercise_selection",
            "progression",
            "recovery",
        ]
    ] = Field(min_length=1)

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
            "endurance",
        ]
    ] = Field(min_length=1)

    applies_to: list[
        Literal[
            "all",
            "hypertrophy",
            "strength",
            "fat_loss",
            #"endurance",
        ]
    ] = Field(min_length=1)

    knowledge_type: list[
        Literal[
            "definition",
            "principle",
            "recommendation",
            "warning",
            "protocol",
        ]
    ] = Field(min_length=1)

class GoalNamespaceMetadata(BaseModel):

    muscle: list[
        Literal[
            "all",
            "chest",
            "back",
            "shoulders",
            "biceps",
            "triceps",
            "forearms",
            "quads",
            "hamstrings",
            "glutes",
            "calves",
            "abs",
        ]
    ] = Field(min_length=1)

    topic: list[
        Literal[
            # Science
            "muscle_physiology",
            "neuromuscular_system",
            "muscle_activation",
            "biomechanics",

            # Goal-specific
            "muscle_growth_mechanisms",
            "hypertrophy_programming",
            "maximal_strength",
            "force_production",
            "power_development",
            "neural_adaptation",

            # Programming
            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "exercise_order",
            "periodization",
            "recovery",
            "fatigue_management",
            "advanced_techniques",
        ]
    ] = Field(min_length=1)

    experience_level: Literal[
        "all",
        "beginner",
        "intermediate",
        "advanced",
    ] = Field(min_length=1)

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
        ]
    ] = Field(min_length=1)


class CollectionRoute(BaseModel):
    collection: Literal[
        "principles",
        "hypertrophy",
        "strength"
    ]

class Chunk(BaseModel):
    id: str
    
    text: str

    book: str
    chapter: str
    collection: Literal[
    "principles",
    "hypertrophy",
    "strength",
]

    chunk_index: int

    metadata: PrinciplesMetadata | GoalNamespaceMetadata | None = None


In [3]:
from dataclasses import dataclass

@dataclass
class ChapterInfo:
    title: str
    start_page: int
    end_page: int


@dataclass
class BookInfo:
    title: str
    author: str
    slug: str

@dataclass
class BookPaths:
    book_dir: Path
    chapters_dir: Path
    markdown_dir: Path
    chunks_dir: Path

import re

def slugify(text: str) -> str:
    text = text.lower().strip()

    text = re.sub(r"[^\w\s-]", "", text)

    text = re.sub(r"[-\s]+", "_", text)

    return text

import fitz
book_path = "books/Science-and-development-of-muscle-hypertrophy-by-Brad-Schoenfeld-z-lib.org_.pdf"


def extract_book_info(pdf_path: str) -> BookInfo:
    doc = fitz.open(pdf_path)

    metadata = doc.metadata

    title = metadata.get("title", "").strip()
    author = metadata.get("author", "").strip().rstrip(";")

    doc.close()
    return BookInfo(
        title=title,
        author=author,
        slug=slugify(title),
    )

book = extract_book_info(book_path)

print(book)

def create_book_folders(
    root_dir: Path,
    book: BookInfo,
) -> dict[str, Path]:
    """
    Create the folder structure for a book.

    Structure:
    data/
    └── books/
        └── <book_slug>/
            ├── chapters/
            └── markdown/
    """

    book_dir = root_dir / book.slug

    chapters_dir = book_dir / "chapters"
    markdown_dir = book_dir / "markdown"
    chunks_dir = book_dir / "chunks"


    chapters_dir.mkdir(parents=True, exist_ok=True)
    markdown_dir.mkdir(parents=True, exist_ok=True)
    chunks_dir.mkdir(parents=True, exist_ok=True)

    return BookPaths(
        book_dir=book_dir,
        chapters_dir=chapters_dir,
        markdown_dir=markdown_dir,
        chunks_dir=chunks_dir,
    )

paths = create_book_folders(
    root_dir=Path("data/books"),
    book=book,
)
print(paths.chapters_dir)
print(paths.markdown_dir)
print(paths.chunks_dir)

BookInfo(title='Science and Development of Muscle Hypertrophy', author='Brad Schoenfeld', slug='science_and_development_of_muscle_hypertrophy')
data/books/science_and_development_of_muscle_hypertrophy/chapters
data/books/science_and_development_of_muscle_hypertrophy/markdown
data/books/science_and_development_of_muscle_hypertrophy/chunks


In [4]:
def load_chunk(chunk_path: Path) -> Chunk:
    with open(chunk_path, "r", encoding="utf-8") as f:
        return Chunk.model_validate_json(f.read())
    

def load_all_chunks(
    chunks_dir: Path,
) -> list[Chunk]:

    chunk_paths = sorted(
        chunks_dir.glob("*.json")
    )

    return [
        load_chunk(path)
        for path in tqdm(chunk_paths)
    ]

all_chunks = load_all_chunks(
    paths.chunks_dir,
)

print(len(all_chunks))


  0%|          | 0/1221 [00:00<?, ?it/s]

1221


In [5]:
from pathlib import Path

from sentence_transformers import SentenceTransformer, CrossEncoder


MODELS_DIR = Path("assets/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


MODELS = {
    "bge-m3": (
        SentenceTransformer,
        "BAAI/bge-m3",
    ),
    "bge-reranker-v2-m3": (
        CrossEncoder,
        "BAAI/bge-reranker-v2-m3",
    ),
}


for model_name, (model_class, huggingface_name) in MODELS.items():

    model_path = MODELS_DIR / model_name

    if model_path.exists():
        print(f"{model_name} already exists.")
        continue

    print(f"⬇️ Downloading {huggingface_name}...")

    model = model_class(
        huggingface_name,
        device="cpu",
    )

    model.save(
        str(model_path),
    )

    print(f"Saved to {model_path}")

bge-m3 already exists.
bge-reranker-v2-m3 already exists.


In [6]:
import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [ ]:
dense_model = SentenceTransformer(
    "assets/models/bge-m3"
)

reranker = CrossEncoder(
    "assets/models/bge-reranker-v2-m3"
)

In [7]:
from collections import defaultdict

chunks_by_collection = defaultdict(list)

for chunk in all_chunks:
    chunks_by_collection[chunk.collection].append(chunk)

for collection, chunks in chunks_by_collection.items():
    print(collection, len(chunks))

hypertrophy 1221


In [8]:
corpus_by_collection = {
    collection: [
        chunk.text
        for chunk in chunks
    ]
    for collection, chunks in chunks_by_collection.items()
}

In [9]:
BM25_DIR = Path("data/indices/bm25")
BM25_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


In [7]:
from fastembed import SparseTextEmbedding

FASTEMBED_DIR = Path("assets/models/fastembed")
FASTEMBED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

bm25 = SparseTextEmbedding(
    model_name="Qdrant/bm25",
    cache_dir=str(FASTEMBED_DIR),
)



In [11]:
import fastembed
print(fastembed.__version__)

0.8.0


In [8]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(
    path="data/qdrant",
)

COLLECTIONS = [
    "principles",
    "hypertrophy",
    "strength",
]



In [9]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance,
)


def create_collections(
    client: QdrantClient,
    collections: list[str],
) -> None:

    existing = {
        c.name
        for c in client.get_collections().collections
    }

    for collection in collections:

        if collection in existing:
            print(f"{collection} already exists.")
            continue

        client.create_collection(
            collection_name=collection,

            vectors_config={
                "dense": VectorParams(
                    size=1024,
                    distance=Distance.COSINE,
                ),
            },

            sparse_vectors_config={
                "sparse": SparseVectorParams(),
            },
        )

        print(f"Created {collection}")

In [14]:
create_collections(
    client=qdrant,
    collections=COLLECTIONS,
)

principles already exists.
hypertrophy already exists.
strength already exists.


In [10]:
from itertools import islice
from collections.abc import Iterator


def batch_iterator(
    items: list,
    batch_size: int,
) -> Iterator[list]:

    iterator = iter(items)

    while batch := list(islice(iterator, batch_size)):
        yield batch


In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

def get_chunk_texts(
    chunks: list[Chunk],
) -> list[str]:

    return [
        chunk.text
        for chunk in chunks
    ]

def embed_dense_batch(
    chunks: list[Chunk],
    model: SentenceTransformer,
) -> np.ndarray:

    return model.encode(
        get_chunk_texts(chunks),
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

In [40]:
dense_model = SentenceTransformer(
    "assets/models/bge-m3"
)

INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from assets/models/bge-m3.
The tokenizer you are loading from 'assets/models/bge-m3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [18]:
first_batch = next(
    batch_iterator(
        all_chunks,
        batch_size=64,
    )
)

dense_vectors = embed_dense_batch(
    first_batch,
    dense_model,
)

print(dense_vectors.shape)

(64, 1024)


In [13]:
def embed_sparse_batch(
    chunks: list[Chunk],
    model: SparseTextEmbedding,
):
    return list(
        model.embed(
            get_chunk_texts(chunks),
        )
    )

In [20]:
sparse_vectors = embed_sparse_batch(
    first_batch,
    bm25,
)

print(len(sparse_vectors))
print(type(sparse_vectors[0]))

64
<class 'fastembed.sparse.sparse_embedding_base.SparseEmbedding'>


In [21]:
first_sparse = sparse_vectors[0]

print(first_sparse.indices[:10])
print(first_sparse.values[:10])

[ 920019564 1810453357   97321696 1338150097 1050980247  540174517
 2028158887 1425104078  691807284   65094886]
[1.77469671 1.48719303 1.77469671 1.77469671 1.77469671 1.89693499
 1.77469671 1.77469671 1.48719303 1.48719303]


In [ ]:
del dense_model
import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [11]:
from qdrant_client.models import (
    PointStruct,
    SparseVector,
)
import uuid


In [12]:
from qdrant_client.models import (
    PointStruct,
    SparseVector,
)
import uuid


def build_points(
    chunks: list[Chunk],
    dense_vectors,
    sparse_vectors,
) -> list[PointStruct]:

    points = []

    for chunk, dense, sparse in zip(
        chunks,
        dense_vectors,
        sparse_vectors,
    ):
        point_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_DNS,
                chunk.id,
            )
        )
        point = PointStruct(
            id=point_id,
            vector={
                "dense": dense.tolist(),
                "sparse": SparseVector(
                    indices=sparse.indices.tolist(),
                    values=sparse.values.tolist(),
                ),
            },
            payload = chunk.model_dump(
                exclude={
                    "collection",
                },
            ),
        )

        points.append(point)

    return points

In [25]:
points = build_points(
    chunks=first_batch,
    dense_vectors=dense_vectors,
    sparse_vectors=sparse_vectors,
)

print(len(points))
print(points[0])

64
id='e3347c7c-fdca-5819-bcaf-1681d8a87770' vector={'dense': [0.03303385153412819, -0.0006948559894226491, -0.03358040377497673, -0.008666672743856907, -0.02898256666958332, -0.04126295819878578, 0.01308109425008297, -0.00198165001347661, 0.028363781049847603, -0.027291705831885338, -0.019360220059752464, -0.0014074182836338878, -0.07290428131818771, 0.019104501232504845, -0.018510300666093826, -0.039340462535619736, -0.0036146908532828093, -0.03405064344406128, -0.002017437480390072, -0.0026961378753185272, 0.010799573734402657, -0.031627509742975235, 0.042020950466394424, 0.0313098207116127, -0.021788988262414932, 0.04102179408073425, 0.0021698204800486565, 0.04287342727184296, 0.014122811146080494, 0.02757268026471138, 0.010984702035784721, 0.0049012405797839165, 0.024349939078092575, -0.01007808092981577, 0.0028526398818939924, -0.019377924501895905, 0.013759990222752094, -0.04983675479888916, -0.0212863702327013, 0.0864463523030281, -0.013665569946169853, -0.03869611769914627, 0.

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct


def upsert_batch(
    client: QdrantClient,
    collection_name: str,
    points: list[PointStruct],
) -> None:

    client.upsert(
        collection_name=collection_name,
        points=points,
        wait=True,
    )
    


In [14]:
from collections import defaultdict
from tqdm import tqdm

def ingest_chunks(
    chunks: list[Chunk],
    client: QdrantClient,
    dense_model: SentenceTransformer,
    sparse_model: SparseTextEmbedding,
    batch_size: int = 64,
) -> None:

    total_batches = (len(chunks) + batch_size - 1) // batch_size

    for batch in tqdm(
        batch_iterator(chunks, batch_size),
        total=total_batches,
        desc="Ingesting Chunks",
    ):
        
        dense_vectors = embed_dense_batch(
            chunks=batch,
            model=dense_model,
        )

        sparse_vectors = embed_sparse_batch(
            chunks=batch,
            model=sparse_model,
        )

        grouped = defaultdict(list)

        for chunk, dense, sparse in zip(
            batch,
            dense_vectors,
            sparse_vectors,
        ):
            grouped[chunk.collection].append(
                (
                    chunk,
                    dense,
                    sparse,
                )
            )

        for collection_name, items in grouped.items():

            group_chunks = [item[0] for item in items]
            group_dense = [item[1] for item in items]
            group_sparse = [item[2] for item in items]

            points = build_points(
                chunks=group_chunks,
                dense_vectors=group_dense,
                sparse_vectors=group_sparse,
            )

            upsert_batch(
                client=client,
                collection_name=collection_name,
                points=points,
            )

In [31]:
ingest_chunks(
    chunks=all_chunks,
    client=qdrant,
    dense_model=dense_model,
    sparse_model=bm25,
    batch_size=64,
)

Ingesting Chunks: 100%|██████████| 20/20 [00:53<00:00,  2.66s/it]


In [32]:
for collection in COLLECTIONS:

    count = qdrant.count(
        collection_name=collection,
        exact=True,
    )

    print(
        f"{collection}: {count.count}"
    )

principles: 0
hypertrophy: 1221
strength: 0


In [33]:
result = qdrant.scroll(
    collection_name="hypertrophy",
    limit=1,
    with_payload=True,
    with_vectors=True,
)

result

([Record(id='002caa39-6ef6-53a3-be97-ce11f65b2f6c', payload={'id': 'science_and_development_of_muscle_hypertrophy_000621', 'text': 'Current evidence suggests little difference in muscle hypertrophy when training with isotonic repetition durations ranging from 0.5 to 6 seconds to muscular failure. Thus, it would seem that a fairly wide range of repetition durations can be used if the primary goal is to maximize muscle growth. Research is limited on the topic, making it difficult to draw concrete conclusions. Concentric tempos of 1 to 3 seconds can be considered viable options; an eccentric tempo of at least 2 seconds appears necessary to ensure loads are lowered under muscular control. On the other hand, training at very slow volitional durations (&gt;10 seconds per repetition) appears to produce inferior increases in muscle growth, although a lack of controlled studies on the topic makes it difficult to draw', 'book': 'Science and Development of Muscle Hypertrophy', 'chapter': 'Chapter

In [34]:
point = result[0][0]

len(
    point.vector["dense"]
)

1024

In [35]:
point.vector["sparse"]

SparseVector(indices=[19522071, 97321696, 131004638, 150760872, 223131028, 259182696, 264741300, 267423339, 301705055, 327797310, 330588669, 372734888, 407983593, 436751995, 446656910, 487501795, 520409122, 522609393, 551536775, 563445918, 608954612, 614428231, 640124220, 662216322, 670727360, 679262712, 724849589, 763520878, 764297089, 769443847, 825859403, 834323496, 897829118, 917062287, 1010544659, 1052767088, 1053731502, 1078715859, 1079573775, 1117092987, 1120066484, 1187906460, 1190790875, 1364383302, 1394226660, 1408635353, 1434869899, 1441330805, 1445293717, 1480150064, 1495971352, 1543434194, 1559397529, 1612148533, 1623065834, 1660786075, 1670835917, 1717433246, 1732871439, 1807167981, 1810453357, 1903144703, 1951551190, 2031875777], values=[1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.3790401567091088, 1.379040156709108

In [ ]:
point.payload

{'id': 'science_and_development_of_muscle_hypertrophy_000621',
 'text': 'Current evidence suggests little difference in muscle hypertrophy when training with isotonic repetition durations ranging from 0.5 to 6 seconds to muscular failure. Thus, it would seem that a fairly wide range of repetition durations can be used if the primary goal is to maximize muscle growth. Research is limited on the topic, making it difficult to draw concrete conclusions. Concentric tempos of 1 to 3 seconds can be considered viable options; an eccentric tempo of at least 2 seconds appears necessary to ensure loads are lowered under muscular control. On the other hand, training at very slow volitional durations (&gt;10 seconds per repetition) appears to produce inferior increases in muscle growth, although a lack of controlled studies on the topic makes it difficult to draw',
 'book': 'Science and Development of Muscle Hypertrophy',
 'chapter': 'Chapter 4: Role of Resistance Training Variables in Hypertrophy'

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def embed_dense_query(
    query: str,
    model: SentenceTransformer,
) -> list[float]:

    return model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

def embed_sparse_query(
    query: str,
    model: SparseTextEmbedding,
) -> SparseVector:

    sparse = list(
        model.embed([query])
    )[0]

    return SparseVector(
        indices=sparse.indices.tolist(),
        values=sparse.values.tolist(),
    )

query = "How do I build muscle?"

dense_vector = embed_dense_query(
    query,
    dense_model,
)

sparse_vector = embed_sparse_query(
    query,
    bm25,
)

In [78]:
import inspect

print(inspect.signature(qdrant.query_points))

(collection_name: str, query: Union[int, str, uuid.UUID, qdrant_common_pb2.PointId, list[float], list[list[float]], qdrant_client.http.models.models.SparseVector, qdrant_client.http.models.models.NearestQuery, qdrant_client.http.models.models.RecommendQuery, qdrant_client.http.models.models.DiscoverQuery, qdrant_client.http.models.models.ContextQuery, qdrant_client.http.models.models.OrderByQuery, qdrant_client.http.models.models.FusionQuery, qdrant_client.http.models.models.RrfQuery, qdrant_client.http.models.models.FormulaQuery, qdrant_client.http.models.models.SampleQuery, qdrant_client.http.models.models.RelevanceFeedbackQuery, numpy.ndarray[tuple[Any, ...], numpy.dtype[Union[numpy.bool, numpy.int8, numpy.int16, numpy.int32, numpy.int64, numpy.uint8, numpy.uint16, numpy.uint32, numpy.uint64, numpy.float16, numpy.float32, numpy.float64, numpy.longdouble]]], qdrant_client.http.models.models.Document, qdrant_client.http.models.models.Image, qdrant_client.http.models.models.InferenceOb

In [15]:
from qdrant_client.models import (
    Prefetch,
    FusionQuery,
    Fusion,
    Filter,
)

def hybrid_search(
    client: QdrantClient,
    collection_name: str,
    query: str,
    dense_model: SentenceTransformer,
    sparse_model: SparseTextEmbedding,
    top_k: int = 10,
    query_filter: Filter | None = None,
):

    dense_vector = embed_dense_query(
        query=query,
        model=dense_model,
    )

    sparse_vector = embed_sparse_query(
        query=query,
        model=sparse_model,
    )

    return client.query_points(
        collection_name=collection_name,

        prefetch=[
            Prefetch(
                using="dense",
                query=dense_vector,
                limit=top_k,
            ),
            Prefetch(
                using="sparse",
                query=sparse_vector,
                limit=top_k,
            ),
        ],

        query=FusionQuery(
            fusion=Fusion.RRF,
        ),

        query_filter=query_filter,

        limit=top_k,
        with_payload=True,
    )

In [39]:
results = hybrid_search(
    client=qdrant,
    collection_name="hypertrophy",
    query="user is having backpain what is the best exercises for him?",
    dense_model=dense_model,
    sparse_model=bm25,
    top_k=10,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [40]:
for point in results.points:

    print(point.score)
    print(point.payload["chapter"])
    print(point.payload["text"][:250])
    print("-" * 80)

0.5
Chapter 8: Program Design for Maximal Hypertrophy
The midback muscles (middle trapezius and rhomboids) are best targeted using sagittal plane exercises (e.g., bent-over row and seated row). A neutral grip reduces biceps brachii activation, which seemingly allows the back musculature to carry out a g
--------------------------------------------------------------------------------
0.5
Chapter 8: Program Design for Maximal Hypertrophy
As noted in a recent review (121), practitioners are best served by viewing set and volume prescription for single- and multi-joint exercises on a 1:1 basis, and then using logical rationale and personal expertise to guide exercise program design. Wh
--------------------------------------------------------------------------------
0.375
Chapter 8: Program Design for Maximal Hypertrophy
activation occurs in the lengthened position, whereas an exercise such as the hip thrust is best for optimizing mechanical tension. Indeed, research shows that the gluteus 

In [41]:
from collections import defaultdict

chapters: dict[str, list[Chunk]] = defaultdict(list)

for chunk in all_chunks:
    chapters[chunk.chapter].append(chunk)

In [42]:
for chapter in chapters:
    print(chapter)

Chapter 1: Hypertrophy-Related Responses and Adaptations to Exercise Stress
Chapter 2: Mechanisms of Hypertrophy
Chapter 3: The Measurement of Muscle Hypertrophy
Chapter 4: Role of Resistance Training Variables in Hypertrophy
Chapter 5: Advanced Training Practices
Chapter 6: Role of Aerobic Training in Hypertrophy
Chapter 7: Factors in Maximal Hypertrophic Development
Chapter 8: Program Design for Maximal Hypertrophy
Chapter 9: Nutrition for Hypertrophy


In [43]:
from pathlib import Path

chapter_names = list(chapters.keys())
idx = 7

chapter_name = chapter_names[idx]
selected_chapter = chapters[chapter_name]

lines = [f"# {chapter_name}", ""]

for chunk in sorted(selected_chapter, key=lambda c: c.chunk_index):
    lines.extend([
        "---",
        "",
        f"## Chunk {chunk.chunk_index}",
        "",
        f"**ID:** `{chunk.id}`",
        "",
        f"**Book:** {chunk.book}",
        "",
        f"**Collection:** {chunk.collection}",
        "",
        "### Text",
        "",
        chunk.text,
        "",
    ])

Path(f"{chapter_name}.md").write_text(
    "\n".join(lines),
    encoding="utf-8",
)

print(f"Saved to {chapter_name}.md")

Saved to Chapter 8: Program Design for Maximal Hypertrophy.md


In [16]:
from typing import Iterable


def hit_rate(
    ground_truth: set[int],
    retrieved: list[int],
) -> float:
    """
    Computes Hit Rate for a single query.

    Returns:
        1.0 if at least one relevant chunk is retrieved.
        0.0 otherwise.
    """
    return float(bool(ground_truth.intersection(retrieved)))

def negative_success_rate(
    retrieved: list[int],
) -> float:
    """
    Computes Negative Success Rate for a single negative query.

    Returns:
        1.0 if no chunks are retrieved.
        0.0 otherwise.
    """
    return float(len(list(retrieved)) == 0)


def precision_at_k(
    ground_truth: set[int],
    retrieved: list[int],
) -> float:
    """
    Computes Precision@k for a single query.

    Precision@k = (# relevant retrieved) / (# retrieved)
    """
    retrieved = list(retrieved)

    if not retrieved:
        return 0.0

    relevant_retrieved = len(
        ground_truth.intersection(retrieved)
    )

    return relevant_retrieved / len(retrieved)

def recall_at_k(
    ground_truth: set[int],
    retrieved: list[int],
) -> float:
    """
    Computes Recall@k for a single query.

    Recall@k = (# relevant retrieved) / (# relevant)
    """

    if not ground_truth:
        return 0.0

    relevant_retrieved = len(
        ground_truth.intersection(retrieved)
    )

    return relevant_retrieved / len(ground_truth)

def reciprocal_rank(
    ground_truth: set[int],
    retrieved: list[int],
) -> float:
    """
    Computes Reciprocal Rank (RR) for a single query.

    RR = 1 / rank of the first relevant retrieved chunk.
    Returns 0.0 if no relevant chunk is retrieved.
    """

    for rank, chunk_idx in enumerate(retrieved, start=1):
        if chunk_idx in ground_truth:
            return 1.0 / rank

    return 0.0




In [17]:
from math import log2


def ndcg(
    ground_truth: set[int],
    retrieved: list[int],
) -> float:
    """
    Computes Normalized Discounted Cumulative Gain (nDCG)
    for a single query.
    """

    if not ground_truth:
        return 0.0

    # DCG
    dcg = 0.0

    for rank, chunk_idx in enumerate(retrieved, start=1):
        if chunk_idx in ground_truth:
            dcg += 1 / log2(rank + 1)

    # IDCG (ideal ranking)
    ideal_hits = min(len(ground_truth), len(retrieved))

    idcg = sum(
        1 / log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [18]:
import json

with open("chapter_8_eval_dataset_v2.json", "r", encoding="utf-8") as f:
    evaluation_dataset = json.load(f)

print(len(evaluation_dataset))

54


In [19]:
from abc import ABC, abstractmethod

class ScoredChunk(Chunk):
    score: float
    
class BaseRetriever(ABC):

    @abstractmethod
    def retrieve(
        self,
        query: str,
        query_filter=None,
    ) -> list[ScoredChunk]:
        """
        Retrieve the most relevant chunk indices
        for a given query.
        """
        pass

In [20]:
import pandas as pd

class RetrieverEvaluator:

    POSITIVE_METRICS = {
        "Precision@k": "precision",
        "Recall@k": "recall",
        "MRR": "reciprocal_rank",
        "nDCG": "ndcg",
        "Hit Rate": "hit_rate",
    }

    NEGATIVE_METRICS = {
        "Negative Success Rate": "negative_success",
    }


    def __init__(
        self,
        retriever: BaseRetriever,
        evaluation_dataset: list[dict],
    ):
        self.retriever = retriever
        self.dataset = evaluation_dataset

        self.results = pd.DataFrame()


    @property
    def positive_results(self) -> pd.DataFrame:
        return self.results[
            self.results["type"] != "negative"
        ]


    @property
    def negative_results(self) -> pd.DataFrame:
        return self.results[
            self.results["type"] == "negative"
        ]

    def _check_results(self) -> None:
        if self.results.empty:
            raise ValueError(
                "Run evaluate() before requesting reports."
            )
        

    def _evaluate_sample(
        self,
        sample: dict,
    ) -> dict:

        question = sample["question"]
        ground_truth = set(sample["relevant_chunks"])
        query_type = sample["type"]

        retrieved_chunks = self.retriever.retrieve(question)

        retrieved_ids = [
            chunk.chunk_index
            for chunk in retrieved_chunks
        ]

        row = {
            "question": question,
            "type": query_type,

            "ground_truth": sorted(ground_truth),

            "retrieved": retrieved_ids,
            "retrieved_chunks": retrieved_chunks,

            "num_ground_truth": len(ground_truth),
            "num_retrieved": len(retrieved_ids),
        }

        if query_type == "negative":

            row["negative_success"] = negative_success_rate(
                retrieved=retrieved_ids,
            )

        else:

            row["hit_rate"] = hit_rate(
                ground_truth=ground_truth,
                retrieved=retrieved_ids,
            )

            row["precision"] = precision_at_k(
                ground_truth=ground_truth,
                retrieved=retrieved_ids,
            )

            row["recall"] = recall_at_k(
                ground_truth=ground_truth,
                retrieved=retrieved_ids,
            )

            row["reciprocal_rank"] = reciprocal_rank(
                ground_truth=ground_truth,
                retrieved=retrieved_ids,
            )

            row["ndcg"] = ndcg(
                ground_truth=ground_truth,
                retrieved=retrieved_ids,
            )

        return row
    

    def evaluate(
        self,
    ) -> pd.DataFrame:

        rows = []

        for sample in self.dataset:
            rows.append(
                self._evaluate_sample(sample)
            )

        self.results = pd.DataFrame(rows)

        return self.results
    

    def summary(
        self,
    ) -> pd.DataFrame:

        self._check_results()

        rows = []

        for display_name, column in self.POSITIVE_METRICS.items():

            rows.append(
                {
                    "Metric": display_name,
                    "Value": self.positive_results[column].mean(),
                }
            )

        for display_name, column in self.NEGATIVE_METRICS.items():

            rows.append(
                {
                    "Metric": display_name,
                    "Value": self.negative_results[column].mean(),
                }
            )

        return pd.DataFrame(rows)
    
    def summary_by_type(
        self,
    ) -> pd.DataFrame:

        self._check_results()

        rows = []

        for query_type, group in self.results.groupby("type"):

            row = {
                "Type": query_type,
                "Count": len(group),
            }

            if query_type == "negative":

                for display_name, column in self.NEGATIVE_METRICS.items():
                    row[display_name] = group[column].mean()

            else:

                for display_name, column in self.POSITIVE_METRICS.items():
                    row[display_name] = group[column].mean()

            rows.append(row)

        return pd.DataFrame(rows)

In [21]:


class HybridRetriever(BaseRetriever):

    def __init__(
        self,
        client: QdrantClient,
        collection_name: str,
        dense_model: SentenceTransformer,
        sparse_model: SparseTextEmbedding,
        top_k: int = 10,
    ):
        self.client = client
        self.collection_name = collection_name
        self.dense_model = dense_model
        self.sparse_model = sparse_model
        self.top_k = top_k

    def retrieve(
        self,
        query: str,
        query_filter: Filter | None = None,
    ) -> list[ScoredChunk]:

        results = hybrid_search(
            client=self.client,
            collection_name=self.collection_name,
            query=query,
            dense_model=self.dense_model,
            sparse_model=self.sparse_model,
            top_k=self.top_k,
            query_filter=query_filter,
        )


        retrieved_chunks = []

        for point in results.points:

            retrieved_chunks.append(
                ScoredChunk(
                    **point.payload,
                    collection=self.collection_name,
                    score=point.score,
                )
            )

        return retrieved_chunks
    
    

In [73]:
hybrid_retriever = HybridRetriever(
    client=qdrant,
    collection_name="hypertrophy",
    dense_model=dense_model,
    sparse_model=bm25,
)

In [ ]:
evaluator = RetrieverEvaluator(
    retriever=hybrid_retriever,
    evaluation_dataset=evaluation_dataset,
)

In [ ]:
results = evaluator.evaluate()

In [53]:
results.head()

,question,type,ground_truth,retrieved,retrieved_chunks,num_ground_truth,num_retrieved,hit_rate,precision,recall,reciprocal_rank,ndcg,negative_success
0,Why can a muscle actually get stronger when it...,single_chunk,[4],"[11, 199, 4, 200, 52, 79, 2, 10, 5, 139]",[id='science_and_development_of_muscle_hypertr...,1,10,1.0,0.1,1.0,0.333333,0.5,NaN
1,Is there a way exercises get grouped based on ...,single_chunk,[18],"[181, 185, 182, 184, 31, 183, 20, 103, 77, 108]",[id='science_and_development_of_muscle_hypertr...,1,10,0.0,0.0,0.0,0.000000,0.0,NaN
2,"Does it matter what angle I do an exercise at,...",single_chunk,[12],"[12, 92, 16, 90, 64, 144, 163, 45, 64, 148]",[id='science_and_development_of_muscle_hypertr...,1,10,1.0,0.1,1.0,1.000000,1.0,NaN
3,What are the different directions my body can ...,single_chunk,[13],"[13, 70, 12, 93, 14, 127, 69, 183, 92, 7]",[id='science_and_development_of_muscle_hypertr...,1,10,1.0,0.1,1.0,1.000000,1.0,NaN
4,Does it really matter how wide I put my hands ...,single_chunk,[15],"[15, 64, 148, 199, 3, 14, 6, 105, 88, 200]",[id='science_and_development_of_muscle_hypertr...,1,10,1.0,0.1,1.0,1.000000,1.0,NaN


In [54]:
evaluator.summary()

,Metric,Value
0,Precision@k,0.109091
1,Recall@k,0.674242
2,MRR,0.571203
3,nDCG,0.548918
4,Hit Rate,0.818182
5,Negative Success Rate,0.000000


In [55]:
evaluator.summary_by_type()

,Type,Count,Precision@k,Recall@k,MRR,nDCG,Hit Rate,Negative Success Rate
0,multi_chunk,17,0.158824,0.509804,0.591503,0.451948,0.882353,NaN
1,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
2,single_chunk,27,0.077778,0.777778,0.558422,0.609973,0.777778,NaN


In [22]:
from abc import ABC, abstractmethod
from typing import Generic, TypeVar

T = TypeVar("T")


class BaseMetadataExtractor(ABC, Generic[T]):

    @abstractmethod
    def extract(
        self,
        query: str,
    ) -> T:
        """
        Extract metadata from a user query.
        """
        pass

In [23]:
from pydantic import BaseModel
from typing import Literal


class PrinciplesQueryFilter(BaseModel):

    topic: list[
        Literal[
            "program_design",
            "periodization",
            "progressive_overload",
            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "recovery",
            "fatigue",
            "warmup",
            "energy_systems",
        ]
    ] | None = None

    planner_stage: list[
        Literal[
            "goal_selection",
            "program_design",
            "exercise_selection",
            "progression",
            "recovery",
        ]
    ] | None = None

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
            "endurance",
        ]
    ] | None = None

    applies_to: list[
        Literal[
            "all",
            "hypertrophy",
            "strength",
            "fat_loss",
        ]
    ] | None = None

    knowledge_type: list[
        Literal[
            "definition",
            "principle",
            "recommendation",
            "warning",
            "protocol",
        ]
    ] | None = None


class GoalQueryFilter(BaseModel):

    muscle: list[
        Literal[
            "all",
            "chest",
            "back",
            "shoulders",
            "biceps",
            "triceps",
            "forearms",
            "quads",
            "hamstrings",
            "glutes",
            "calves",
            "abs",
        ]
    ] | None = None

    topic: list[
        Literal[
            "muscle_physiology",
            "neuromuscular_system",
            "muscle_activation",
            "biomechanics",

            "muscle_growth_mechanisms",
            "hypertrophy_programming",
            "maximal_strength",
            "force_production",
            "power_development",
            "neural_adaptation",

            "volume",
            "frequency",
            "intensity",
            "load",
            "exercise_selection",
            "exercise_order",
            "periodization",
            "recovery",
            "fatigue_management",
            "advanced_techniques",
        ]
    ] | None = None

    experience_level: Literal[
        "all",
        "beginner",
        "intermediate",
        "advanced",
    ] | None = None

    goals: list[
        Literal[
            "hypertrophy",
            "strength",
            "fat_loss",
        ]
    ] | None = None

In [24]:
from langchain_core.prompts import ChatPromptTemplate


GOAL_QUERY_FILTER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in exercise science.

Your task is to extract structured retrieval filters from a user's query.

The output will be used to filter a vector database before retrieval.

Guidelines:

1. Understand the semantic meaning of the query, not just the exact words.

2. Infer metadata when it is strongly implied by the user's intent.

3. Never guess. If you are not confident about a field, leave it empty.

4. Only extract metadata that is useful for retrieval.

5. Do not force every field to be populated.

6. Do not infer metadata solely from common associations.
   For example, "bench press" does not automatically imply "chest" unless the query is actually about training the chest.

7. Use "all" only when the user explicitly refers to everyone or to general recommendations.

8. Ignore conversational text that does not affect retrieval.

9. Return only the structured output.
            """,
        ),
        (
            "human",
            """
User Query:

{query}
            """,
        ),
    ]
)

In [25]:
from langchain_core.language_models import BaseChatModel


class GoalMetadataExtractor(
    BaseMetadataExtractor[GoalQueryFilter]
):

    def __init__(
        self,
        llm: BaseChatModel,
    ):
        self.chain = (
            GOAL_QUERY_FILTER_PROMPT
            | llm.with_structured_output(GoalQueryFilter)
        )

        self._cache: dict[str, GoalQueryFilter] = {}

    def extract(
        self,
        query: str,
    ) -> GoalQueryFilter:

        if query in self._cache:
            return self._cache[query]

        metadata = self.chain.invoke(
            {
                "query": query,
            }
        )

        self._cache[query] = metadata

        return metadata

In [26]:
from dotenv import load_dotenv

load_dotenv()
os.getenv("HF_HOME")
os.getenv("GOOGLE_API_KEY")
os.getenv("GROQ_API_KEY")
os.getenv("NVIDIA_API_KEY")
os.getenv("NVIDIA_API_KEY")

os.getenv("ITI_API_KEY")
print()

In [27]:
from enum import Enum

class LLMProvider(Enum):
    GEMINI = "gemini"
    GROQ = "groq"
    NVIDIA = "nvidia"
    OPENROUTER = "openrouter"
    ITI = "iti"


@dataclass(frozen=True)
class LLMConfig:
    provider: LLMProvider
    model: str
    temperature: float = 0.0




In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openai import ChatOpenAI

def create_llm(config: LLMConfig) -> BaseChatModel:

    if config.provider == LLMProvider.GEMINI:
        return ChatGoogleGenerativeAI(
            model=config.model,
            temperature=config.temperature,
        )

    if config.provider == LLMProvider.GROQ:
        return ChatGroq(
            model=config.model,
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.NVIDIA:
        return ChatNVIDIA(
            model=config.model,
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.OPENROUTER:
        return ChatOpenAI(
            openai_api_base="https://openrouter.ai/api/v1",
            model=config.model,
            api_key=os.getenv("OPENROUTER_API_KEY"),
            temperature=config.temperature,
        )
    
    if config.provider == LLMProvider.ITI:
        return ChatOpenAI(
            base_url="http://apiaccess.iti.net.eg/api/v1",
            model=config.model,
            temperature=config.temperature,
        )

    raise ValueError(
        f"Unsupported provider: {config.provider}"
    )

In [34]:

config = LLMConfig(
    provider=LLMProvider.GROQ,
    model="openai/gpt-oss-120b",
)

llm = create_llm(config)

In [29]:
T = TypeVar("T")

class BaseFilterBuilder(ABC, Generic[T]):

    @abstractmethod
    def build(
        self,
        query_filter: T,
    ):
        pass

In [30]:
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchAny,
    MatchValue,
)


class GoalFilterBuilder(
    BaseFilterBuilder[GoalQueryFilter]
):

    def build(
        self,
        query_filter: GoalQueryFilter,
    ) -> Filter:

        must = []

        if query_filter.muscle:
            must.append(
                FieldCondition(
                    key="metadata.muscle",
                    match=MatchAny(
                        any=[
                            *query_filter.muscle,
                            "all",
                        ],
                    ),
                )
            )

        if query_filter.topic:
            must.append(
                FieldCondition(
                    key="metadata.topic",
                    match=MatchAny(
                        any=query_filter.topic,
                    ),
                )
            )

        if query_filter.experience_level:
            must.append(
                FieldCondition(
                    key="metadata.experience_level",
                    match=MatchAny(
                        any=[
                            query_filter.experience_level,
                            "all",
                        ],
                    ),
                )
            )

        if query_filter.goals:
            must.append(
                FieldCondition(
                    key="metadata.goals",
                    match=MatchAny(
                        any=query_filter.goals,
                    ),
                )
            )

        return Filter(
            must=must,
        )

In [87]:
query_filter = GoalQueryFilter(
    muscle=["chest"],
    topic=["volume", "intensity"],
    experience_level="beginner",
    goals=["hypertrophy"],
)

builder = GoalFilterBuilder()

builder.build(query_filter)

Filter(should=None, min_should=None, must=[FieldCondition(key='metadata.muscle', match=MatchAny(any=['chest', 'all']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.topic', match=MatchAny(any=['volume', 'intensity']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.experience_level', match=MatchAny(any=['beginner', 'all']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.goals', match=MatchAny(any=['hypertrophy']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)], must_not=None)

In [31]:
class FilteredHybridRetriever(BaseRetriever):

    def __init__(
        self,
        retriever: HybridRetriever,
        metadata_extractor: GoalMetadataExtractor,
        filter_builder: GoalFilterBuilder,
    ):
        self.retriever = retriever
        self.metadata_extractor = metadata_extractor
        self.filter_builder = filter_builder

    def retrieve(
        self,
        query: str,
    ) -> list[ScoredChunk]:

        query_filter = self.metadata_extractor.extract(query)

        qdrant_filter = self.filter_builder.build(query_filter)

        return self.retriever.retrieve(
            query=query,
            query_filter=qdrant_filter,
        )

In [93]:
hybrid_retriever = HybridRetriever(
    client=qdrant,
    collection_name="hypertrophy",
    dense_model=dense_model,
    sparse_model=bm25,
)

metadata_extractor = GoalMetadataExtractor(
    llm=llm,
)

filter_builder = GoalFilterBuilder()

filtered_retriever = FilteredHybridRetriever(
    retriever=hybrid_retriever,
    metadata_extractor=metadata_extractor,
    filter_builder=filter_builder,
)

In [90]:
results = filtered_retriever.retrieve(
    "How many sets should beginners perform for chest hypertrophy?"
)

len(results)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

10

In [91]:
query_filter = metadata_extractor.extract(
    "How many sets should beginners perform for chest hypertrophy?"
)

print(query_filter)

print(filter_builder.build(query_filter))

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


muscle=['chest'] topic=['volume'] experience_level='beginner' goals=['hypertrophy']
should=None min_should=None must=[FieldCondition(key='metadata.muscle', match=MatchAny(any=['chest', 'all']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.topic', match=MatchAny(any=['volume']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.experience_level', match=MatchAny(any=['beginner', 'all']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='metadata.goals', match=MatchAny(any=['hypertrophy']), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None


In [92]:
results = filtered_retriever.retrieve(
    "How many sets should beginners perform for chest hypertrophy?"
)

results[0]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ScoredChunk(id='science_and_development_of_muscle_hypertrophy_000675', text='- Multiset protocols favoring higher volumes of resistance training optimize the hypertrophic response. A range of 10 to 20 sets per muscle is a general guideline for weekly volume prescription. That said, there is a fairly wide interindividual response to the volume dose, and thus some people will thrive on somewhat lower volumes, while others will benefit from slightly higher volumes. Strategic use of very high volumes (~30+ sets per muscle) can be employed to help bring up lagging muscle groups. To avoid overtraining, overall volume should be progressively increased over the course of a training cycle; periods of reduced training volume should be integrated regularly to facilitate the recovery process.', book='Science and Development of Muscle Hypertrophy', chapter='Chapter 4: Role of Resistance Training Variables in Hypertrophy', collection='hypertrophy', chunk_index=223, metadata=GoalNamespaceMetadata(mus

In [32]:


class RetrieverExperimentRunner:

    def __init__(
        self,
        evaluation_dataset: list[dict],
    ):
        self.dataset = evaluation_dataset

        self.summary_results = []
        self.type_results = []

    def run(
        self,
        name: str,
        retriever: BaseRetriever,
    ) -> None:

        evaluator = RetrieverEvaluator(
            retriever=retriever,
            evaluation_dataset=self.dataset,
        )

        evaluator.evaluate()

        summary = evaluator.summary().set_index("Metric")["Value"].to_dict()
        summary["Experiment"] = name

        self.summary_results.append(summary)

        by_type = evaluator.summary_by_type()
        by_type.insert(0, "Experiment", name)

        self.type_results.append(by_type)

    def summary(
        self,
    ) -> pd.DataFrame:

        return (
            pd.DataFrame(self.summary_results)
            .set_index("Experiment")
        )

    def summary_by_type(
        self,
    ) -> pd.DataFrame:

        return pd.concat(
            self.type_results,
            ignore_index=True,
        )

In [ ]:
runner = RetrieverExperimentRunner(
    evaluation_dataset=evaluation_dataset,
)

runner.run(
    name="Hybrid",
    retriever=hybrid_retriever,
)

runner.run(
    name="Hybrid + Metadata",
    retriever=filtered_retriever,
)

runner.summary()

In [95]:
runner.summary_by_type()

,Experiment,Type,Count,Precision@k,Recall@k,MRR,nDCG,Hit Rate,Negative Success Rate
0,Hybrid,multi_chunk,17,0.158824,0.509804,0.591503,0.451948,0.882353,NaN
1,Hybrid,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
2,Hybrid,single_chunk,27,0.077778,0.777778,0.558422,0.609973,0.777778,NaN
3,Hybrid + Metadata,multi_chunk,17,0.158824,0.509804,0.591503,0.451948,0.882353,NaN
4,Hybrid + Metadata,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
5,Hybrid + Metadata,single_chunk,27,0.077778,0.777778,0.558422,0.609973,0.777778,NaN


In [33]:
from abc import ABC, abstractmethod

class BaseReranker(ABC):

    @abstractmethod
    def rerank(
        self,
        query: str,
        chunks: list[ScoredChunk],
    ) -> list[ScoredChunk]:
        """
        Re-rank retrieved chunks according to their relevance to the query.
        """
        ...

In [34]:

class CrossEncoderReranker(BaseReranker):

    def __init__(
        self,
        model_name: str,
        device: str | None = None,
        batch_size: int = 16,
    ):
        self.batch_size = batch_size
        self.model = CrossEncoder(
            model_name,
            device=device,
        )

    def rerank(
        self,
        query: str,
        chunks: list[ScoredChunk],
    ) -> list[ScoredChunk]:

        pairs = [
            (query, chunk.text)
            for chunk in chunks
        ]

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
        )

        reranked_chunks = [
            chunk.model_copy(
                update={"score": float(score)}
            )
            for chunk, score in zip(chunks, scores)
        ]

        return sorted(
            reranked_chunks,
            key=lambda chunk: chunk.score,
            reverse=True,
        )

In [35]:
class RerankingRetriever(BaseRetriever):

    def __init__(
        self,
        retriever: BaseRetriever,
        reranker: BaseReranker,
    ):
        self.retriever = retriever
        self.reranker = reranker

    def retrieve(
        self,
        query: str,
    ) -> list[ScoredChunk]:

        chunks = self.retriever.retrieve(query=query)

        return self.reranker.rerank(
            query=query,
            chunks=chunks,
        )

In [ ]:
hybrid_retriever = HybridRetriever(
    client=qdrant,
    collection_name="hypertrophy",
    dense_model=dense_model,
    sparse_model=bm25,
    top_k=50
)

In [ ]:
reranker = CrossEncoderReranker(
    model_name="assets/models/bge-reranker-v2-m3",
    device="cpu",
)

reranker_hybrid_retriever = RerankingRetriever(
    retriever=hybrid_retriever,  
    reranker=reranker,
)

In [ ]:
results = reranker_hybrid_retriever.retrieve(
    query="best exercises for chest hypertrophy",
)

In [119]:
print(len(results))

50


In [120]:
for chunk in results[:5]:
    print(chunk.score)

0.9473987817764282
0.7512507438659668
0.4921760857105255
0.4690931439399719
0.42785853147506714


In [ ]:
hybrid_results = hybrid_retriever.retrieve(
    query="best exercises for chest hypertrophy",
)

reranked_results = reranker_hybrid_retriever.retrieve(
    query="best exercises for chest hypertrophy",
)

In [122]:
for i in range(10):
    print(
        i + 1,
        hybrid_results[i].score,
        " | ",
        reranked_results[i].score,
        " | ",
        reranked_results[i].text[:80],
    )

1 0.6  |  0.9473987817764282  |  ## Chest

The pectoralis major is maximally activated in the transverse plane us
2 0.5833333333333334  |  0.7512507438659668  |  1. Long-length accentuated force exercises create maximal torque while the prime
3 0.38095238095238093  |  0.4921760857105255  |  ## KEY POINT

Maximal hypertrophy can be best achieved by systematically varying
4 0.3787878787878788  |  0.4690931439399719  |  - Once competency in the basic movement patterns has been established, a variety
5 0.29  |  0.42785853147506714  |  Exercise selection is another factor worthy of consideration when determining re
6 0.26666666666666666  |  0.4120422303676605  |  ## KEY POINT

Once trainees have learned the movement patterns of basic resistan
7 0.25396825396825395  |  0.3975623548030853  |  Taking the body of literature on the topic into account, interference appears to
8 0.25  |  0.32435476779937744  |  The impact of volume may be at least in part frequency dependent. Schwartz and c
9 0.23

In [123]:
for i in range(10):
    print("=" * 80)
    print("Hybrid:")
    print(hybrid_results[i].text[:100])

    print("\nReranked:")
    print(reranked_results[i].text[:100])

Hybrid:
## KEY POINT

Maximal hypertrophy can be best achieved by systematically varying the exercises perfo

Reranked:
## Chest

The pectoralis major is maximally activated in the transverse plane using horizontal adduc
Hybrid:
## Chest

The pectoralis major is maximally activated in the transverse plane using horizontal adduc

Reranked:
1. Long-length accentuated force exercises create maximal torque while the prime movers are stretche
Hybrid:
It should be noted that that per-session training time in the strength group was 70 minutes, whereas

Reranked:
## KEY POINT

Maximal hypertrophy can be best achieved by systematically varying the exercises perfo
Hybrid:
- Once competency in the basic movement patterns has been established, a variety of exercises should

Reranked:
- Once competency in the basic movement patterns has been established, a variety of exercises should
Hybrid:
Exercise selection is another factor worthy of consideration when determining rest intervals for hyp

Rerank

In [42]:
# config = LLMConfig(
#     provider=LLMProvider.NVIDIA,
#     model="nvidia/nemotron-3-super-120b-a12b",
# )

config = LLMConfig(
    provider=LLMProvider.GROQ,
    model="openai/gpt-oss-120b",
)

# config = LLMConfig(
#    provider=LLMProvider.GEMINI,
#    model="models/gemini-2.5-flash"
# )

# config = LLMConfig(
#     provider=LLMProvider.OPENROUTER,
#     model="nvidia/nemotron-3-ultra-550b-a55b:free",
#     temperature=0
# )

# config = LLMConfig(
#     provider=LLMProvider.ITI,
#     model="anthropic.claude-sonnet-4-6",
#     temperature=0
# )

llm = create_llm(config)

In [43]:

hybrid_retriever = HybridRetriever(
    client=qdrant,
    collection_name="hypertrophy",
    dense_model=dense_model,
    sparse_model=bm25,
    top_k=50
)

metadata_extractor = GoalMetadataExtractor(
    llm=llm,
)

filter_builder = GoalFilterBuilder()

filtered_retriever = FilteredHybridRetriever(
    retriever=hybrid_retriever,
    metadata_extractor=metadata_extractor,
    filter_builder=filter_builder,
)

reranker_hybrid_filtered_retriever = RerankingRetriever(
    retriever=filtered_retriever,  
    reranker=reranker,
)

In [128]:
runner = RetrieverExperimentRunner(
    evaluation_dataset=evaluation_dataset,
)

runner.run(
    name="Hybrid",
    retriever=hybrid_retriever,
)

runner.run(
    name="Hybrid + Metadata",
    retriever=filtered_retriever,
)

runner.run(
    name="Hybrid + Reranker",
    retriever=reranker_hybrid_retriever
)

runner.run(
    name=" Metadata + Hybrid + Reranker",
    retriever=reranker_hybrid_filtered_retriever
    
)

runner.summary()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

,Precision@k,Recall@k,MRR,nDCG,Hit Rate,Negative Success Rate
Experiment,,,,,,
Hybrid,0.029545,0.886364,0.580508,0.638371,0.977273,0.0
Hybrid + Metadata,0.029545,0.886364,0.580508,0.638371,0.977273,0.0
Hybrid + Reranker,0.029545,0.886364,0.647340,0.694599,0.977273,0.0
Metadata + Hybrid + Reranker,0.029545,0.886364,0.647340,0.694599,0.977273,0.0


In [129]:
runner.summary_by_type()

,Experiment,Type,Count,Precision@k,Recall@k,MRR,nDCG,Hit Rate,Negative Success Rate
0,Hybrid,multi_chunk,17,0.045882,0.764706,0.589836,0.570509,1.000000,NaN
1,Hybrid,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
2,Hybrid,single_chunk,27,0.019259,0.962963,0.574635,0.681099,0.962963,NaN
3,Hybrid + Metadata,multi_chunk,17,0.045882,0.764706,0.589836,0.570509,1.000000,NaN
4,Hybrid + Metadata,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
5,Hybrid + Metadata,single_chunk,27,0.019259,0.962963,0.574635,0.681099,0.962963,NaN
6,Hybrid + Reranker,multi_chunk,17,0.045882,0.764706,0.603270,0.582211,1.000000,NaN
7,Hybrid + Reranker,negative,10,NaN,NaN,NaN,NaN,NaN,0.0
8,Hybrid + Reranker,single_chunk,27,0.019259,0.962963,0.675087,0.765362,0.962963,NaN
9,Metadata + Hybrid + Reranker,multi_chunk,17,0.045882,0.764706,0.603270,0.582211,1.000000,NaN


In [44]:
class RAGResponse(BaseModel):
    answer: str
    chunks: list[ScoredChunk]

In [45]:

class BaseRAG(ABC):

    @abstractmethod
    def invoke(
        self,
        query: str,
    ) -> RAGResponse:
        ...

In [46]:
from langchain_core.language_models import BaseChatModel
from langchain_core.prompts import ChatPromptTemplate


class RAG(BaseRAG):

    def __init__(
        self,
        retriever: BaseRetriever,
        llm: BaseChatModel,
        prompt: ChatPromptTemplate,
        context_top_k: int = 10,
    ):
        self.retriever = retriever
        self.context_top_k = context_top_k
        self.chain = prompt | llm

    def _build_context(
        self,
        chunks: list[ScoredChunk],
    ) -> str:

        return "\n\n".join(
            chunk.text
            for chunk in chunks
        )

    def invoke(
        self,
        query: str,
    ) -> RAGResponse:

        chunks = self.retriever.retrieve(
            query=query,
        )

        selected_chunks = chunks[: self.context_top_k]

        context = self._build_context(selected_chunks)

        response = self.chain.invoke(
            {
                "query": query,
                "context": context,
            }
        )

        return RAGResponse(
            answer=response.content,
            chunks=selected_chunks,
        )

In [47]:
RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in exercise science.

Answer ONLY using the provided context.

If the context does not contain the answer, say you don't know.

Do not invent information.

Context:
{context}
""",
        ),
        (
            "human",
            "{query}",
        ),
    ]
)

In [50]:
rag = RAG(
    retriever=reranker_hybrid_filtered_retriever,
    llm=llm,
    prompt=RAG_PROMPT,
    context_top_k=5,
)

response = rag.invoke(
    "What is the optimal training volume for hypertrophy?"
)

print(response.answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The literature suggests that, as a general guideline, **about 10 to 20 sets per muscle group each week** is optimal for maximizing hypertrophy. More advanced lifters may benefit from the higher end of that range, and occasional very‑high volumes (≈30 + sets) can be used strategically for lagging muscles, but the 10‑20‑set per‑week range is the recommended baseline.


In [51]:
response = rag.invoke(
    "How do I cook pasta?"
)

print(response.answer)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


I don't know.


In [52]:
evaluation_dataset[0]

{'question': "Why can a muscle actually get stronger when it's stretched a bit past its normal resting length instead of weaker?",
 'relevant_chunks': [4],
 'type': 'single_chunk'}

In [53]:
response = rag.invoke(evaluation_dataset[0]["question"])
response

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


RAGResponse(answer='The muscle’s ability to generate force depends on how the actin and myosin filaments line up inside each sarcomere.\u202fAt the muscle’s normal resting length the overlap of those filaments is already good, so force production is near its peak.\u202fWhen the muscle is stretched a little beyond that resting length (about\u202f125\u202f%–140\u202f% of resting length), the sarcomeres are lengthened just enough that the filaments are pulled closer together again and the sensitivity of the contractile apparatus to calcium rises.\u202fThis “enhanced calcium sensitivity” and the slightly improved geometry increase the chance that cross‑bridges will form, allowing the muscle to produce even more force than at true resting length.\n\nBecause the muscle can generate higher forces when it is trained in that stretched position, the mechanical stress and time‑under‑tension are greater.\u202fThose larger mechanical stimuli trigger stronger anabolic signaling (e.g., higher IGF‑1 l

In [54]:
ragas_sample = {
    "user_input": evaluation_dataset[0]["question"],
    "response": response.answer,
    "retrieved_contexts": [
        chunk.text for chunk in response.chunks
    ]
}

ragas_sample

{'user_input': "Why can a muscle actually get stronger when it's stretched a bit past its normal resting length instead of weaker?",
 'response': 'The muscle’s ability to generate force depends on how the actin and myosin filaments line up inside each sarcomere.\u202fAt the muscle’s normal resting length the overlap of those filaments is already good, so force production is near its peak.\u202fWhen the muscle is stretched a little beyond that resting length (about\u202f125\u202f%–140\u202f% of resting length), the sarcomeres are lengthened just enough that the filaments are pulled closer together again and the sensitivity of the contractile apparatus to calcium rises.\u202fThis “enhanced calcium sensitivity” and the slightly improved geometry increase the chance that cross‑bridges will form, allowing the muscle to produce even more force than at true resting length.\n\nBecause the muscle can generate higher forces when it is trained in that stretched position, the mechanical stress and

In [55]:
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
)
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_372159/1769669787.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_372159/1769669787.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (


In [56]:
type(llm)

langchain_groq.chat_models.ChatGroq

In [57]:
from ragas import evaluate
import inspect

print(inspect.signature(evaluate))

(dataset: 't.Union[Dataset, EvaluationDataset]', metrics: 't.Optional[t.Sequence[Metric]]' = None, llm: 't.Optional[BaseRagasLLM | LangchainLLM]' = None, embeddings: 't.Optional[BaseRagasEmbeddings | BaseRagasEmbedding | LangchainEmbeddings]' = None, experiment_name: 't.Optional[str]' = None, callbacks: 'Callbacks' = None, run_config: 't.Optional[RunConfig]' = None, token_usage_parser: 't.Optional[TokenUsageParser]' = None, raise_exceptions: 'bool' = False, column_map: 't.Optional[t.Dict[str, str]]' = None, show_progress: 'bool' = True, batch_size: 't.Optional[int]' = None, _run_id: 't.Optional[UUID]' = None, _pbar: 't.Optional[tqdm]' = None, return_executor: 'bool' = False, allow_nest_asyncio: 'bool' = True) -> 't.Union[EvaluationResult, Executor]'


In [58]:
from datasets import Dataset

ragas_dataset = Dataset.from_list([ragas_sample])

ragas_dataset

Dataset({
    features: ['user_input', 'response', 'retrieved_contexts'],
    num_rows: 1
})

In [85]:
from ragas.embeddings.base import BaseRagasEmbeddings

class SentenceTransformerRagasEmbeddings(BaseRagasEmbeddings):

    def __init__(self, model, model_name):
        self.client = model
        self.model = model_name

    def embed_query(self, text):
        return self.client.encode(
            text,
            convert_to_numpy=True
        ).tolist()

    def embed_documents(self, texts):
        return self.client.encode(
            texts,
            convert_to_numpy=True
        ).tolist()

    async def aembed_query(self, text):
        return self.embed_query(text)

    async def aembed_documents(self, texts):
        return self.embed_documents(texts)
    

dense_model = SentenceTransformer(
    "assets/models/bge-m3",
    device="cpu"
)

ragas_embeddings = SentenceTransformerRagasEmbeddings(
    dense_model,
    "bge-m3"
)

INFO:sentence_transformers.base.model:Loading SentenceTransformer model from assets/models/bge-m3.
The tokenizer you are loading from 'assets/models/bge-m3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [86]:
del llm

In [87]:
# config = LLMConfig(
#     provider=LLMProvider.NVIDIA,
#     model="nvidia/nemotron-3-super-120b-a12b",
# )

config = LLMConfig(
    provider=LLMProvider.GROQ,
    model="openai/gpt-oss-120b",
)

# config = LLMConfig(
#    provider=LLMProvider.GEMINI,
#    model="models/gemini-2.5-flash"
# )

# config = LLMConfig(
#     provider=LLMProvider.OPENROUTER,
#     model="nvidia/nemotron-3-ultra-550b-a55b:free",
#     temperature=0
# )

# config = LLMConfig(
#     provider=LLMProvider.ITI,
#     model="anthropic.claude-sonnet-4-6",
#     temperature=0
# )

llm = create_llm(config)

In [76]:
type(llm)

langchain_groq.chat_models.ChatGroq

In [77]:
metrics = [
    Faithfulness(),
    # AnswerRelevancy(),
]

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=ragas_embeddings,
    batch_size=1,
)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Batch 1/1:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [78]:
result

{'faithfulness': 1.0000}

In [79]:
from ragas.metrics import AnswerRelevancy

answer_relevancy = AnswerRelevancy(
    llm=llm,
    embeddings=ragas_embeddings,
    strictness=1
)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[answer_relevancy],
    llm=llm,
    embeddings=ragas_embeddings,
    raise_exceptions=True,
)

/tmp/ipykernel_372159/216054670.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [80]:
result

{'answer_relevancy': 0.8598}

In [81]:
ragas_samples = []

for item in evaluation_dataset:
    question = item["question"]

    rag_response = rag.invoke(question)

    ragas_samples.append(
        {
            "user_input": question,
            "response": rag_response.answer,
            "retrieved_contexts": [
                chunk.text for chunk in rag_response.chunks
            ],
        }
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [82]:
ragas_dataset = Dataset.from_list(ragas_samples)

ragas_dataset

Dataset({
    features: ['user_input', 'response', 'retrieved_contexts'],
    num_rows: 54
})

In [89]:
ragas_dataset.to_json("ragas_dataset.json")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

244370

In [90]:
ragas_dataset.save_to_disk("ragas_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/54 [00:00<?, ? examples/s]

In [88]:
metrics = [
    Faithfulness(),
    AnswerRelevancy(
        strictness=1
    ),
]

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=ragas_embeddings,
    batch_size=1,
    raise_exceptions=True,
)

Evaluating:   0%|          | 0/108 [00:00<?, ?it/s]

Batch 1/108:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


KeyboardInterrupt: 

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
